In [1]:
import sys

sys.path.append("..")

import torch
from torch.utils.data import DataLoader

from src.dataset import ECGDataset, get_eval_transform, CLASS_NAMES
from src.model import build_model

# Build dataset and take one real batch
ds = ECGDataset(
    "../data/folds.csv", "..", folds=[0, 1, 2, 3], transform=get_eval_transform()
)
loader = DataLoader(ds, batch_size=8, shuffle=True)
images, labels = next(iter(loader))
print("input:", images.shape)

# Build model (first run downloads ResNet18 weights, about 45 MB)
model = build_model()
model.eval()  # inference mode (matters for BatchNorm, explained later)

with torch.no_grad():  # no gradients needed for a simple forward pass
    logits = model(images)

probs = torch.softmax(logits, dim=1)

print("logits shape:", logits.shape)  # expect [8, 4]
print("sample logits:", logits[0])
print("sample probs :", probs[0])
print("sum of probs :", probs[0].sum())  # expect 1.0
print("predicted    :", CLASS_NAMES[probs[0].argmax().item()])
print("true label   :", CLASS_NAMES[labels[0].item()])

input: torch.Size([8, 3, 256, 352])
logits shape: torch.Size([8, 4])
sample logits: tensor([ 0.1012, -0.2660,  0.1439,  0.4921])
sample probs : tensor([0.2373, 0.1643, 0.2476, 0.3508])
sum of probs : tensor(1.)
predicted    : Normal
true label   : History_of_MI


In [2]:
import torch.nn as nn

# 1) Class weights from the training counts (rare class -> bigger weight)
counts = ds.df["label"].value_counts()
weights = torch.tensor(
    [len(ds) / (len(CLASS_NAMES) * counts[name]) for name in CLASS_NAMES],
    dtype=torch.float32,
)
print("class weights:", dict(zip(CLASS_NAMES, weights.tolist())))

# 2) Loss and optimizer
# CrossEntropyLoss expects raw logits (NOT softmax probabilities)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# 3) Take ONE fixed batch and train on it repeatedly
fixed_images, fixed_labels = next(iter(loader))
model.train()  # training mode (BatchNorm updates its statistics)

for step in range(30):
    optimizer.zero_grad()  # clear old gradients
    logits = model(fixed_images)  # forward pass
    loss = criterion(logits, fixed_labels)  # how wrong are we?
    loss.backward()  # compute gradients
    optimizer.step()  # update weights
    print(f"step {step:2d} | loss {loss.item():.4f}")

class weights: {'Abnormal': 0.5254010558128357, 'History_of_MI': 1.4239130020141602, 'MI': 4.09375, 'Normal': 0.8694690465927124}
step  0 | loss 1.4607
step  1 | loss 0.4647
step  2 | loss 0.2118
step  3 | loss 0.1115
step  4 | loss 0.0654
step  5 | loss 0.0422
step  6 | loss 0.0296
step  7 | loss 0.0220
step  8 | loss 0.0170
step  9 | loss 0.0136
step 10 | loss 0.0111
step 11 | loss 0.0092
step 12 | loss 0.0077
step 13 | loss 0.0066
step 14 | loss 0.0057
step 15 | loss 0.0050
step 16 | loss 0.0044
step 17 | loss 0.0040
step 18 | loss 0.0036
step 19 | loss 0.0033
step 20 | loss 0.0030
step 21 | loss 0.0028
step 22 | loss 0.0026
step 23 | loss 0.0024
step 24 | loss 0.0022
step 25 | loss 0.0021
step 26 | loss 0.0020
step 27 | loss 0.0019
step 28 | loss 0.0018
step 29 | loss 0.0017
